We start from the Navier Stocks equations:
$$\partial_t u +(u \cdot \nabla )u = -\frac{1}{\rho} \nabla p + \nu \Delta u $$
$$ \nabla \cdot u = 0 $$

The goal is not to develop a solution that derive from an analytic simplification of the problem (one reason beeing i am not an expert in fluid dynamic) but rather use my knowledge in generic discretization to simulate this.

Therefore, we have to reconoize that this equation is not trivial to run, because of the constraint of incompressibility.
The most basic method to simulate this is the Chorin projection method.
I wrote some part of the code with the help of Claude assistant, especially the graphic functions and some code performance upgrades.

## References

**Projection method (fractional steps)**
- A. J. Chorin, *Numerical solution of the Navier–Stokes equations*,
  Mathematics of Computation 22 (1968), 745–762.
- R. Temam, *Sur l'approximation de la solution des équations de Navier–Stokes
  par la méthode des pas fractionnaires (II)*, Arch. Rational Mech. Anal. 33 (1969), 377–385.

**Staggered grid (MAC)**
- F. H. Harlow, J. E. Welch, *Numerical calculation of time-dependent viscous incompressible
  flow of fluid with free surface*, Physics of Fluids 8 (1965), 2182–2189.

**Semi-Lagrangian advection (method of characteristics)**
- O. Pironneau, *On the transport-diffusion algorithm and its applications to the
  Navier–Stokes equations*, Numerische Mathematik 38 (1982), 309–332.
- J. Stam, *Stable Fluids*, Proceedings of SIGGRAPH 1999, 121–128.

**FFT-based Poisson solver on a periodic domain**
- J. Stam, *A Simple Fluid Solver Based on the FFT*, Journal of Graphics Tools 6(2) (2001), 43–52.

**Double shear layer test case**
- J. B. Bell, P. Colella, H. M. Glaz, *A second-order projection method for the incompressible
  Navier–Stokes equations*, Journal of Computational Physics 85 (1989), 257–283.

**Obstacle (inspired by penalization methods)**
- P. Angot, C.-H. Bruneau, P. Fabrie, *A penalization method to take into account obstacles
  in incompressible viscous flows*, Numerische Mathematik 81 (1999), 497–520.

**Further reading**
- R. Bridson, *Fluid Simulation for Computer Graphics*, 2nd ed., CRC Press, 2015
  (MAC grid, semi-Lagrangian advection and pressure projection: the same structure as this code).
- L. A. Barba, G. F. Forsyth, *CFD Python: the 12 steps to Navier–Stokes equations*,
  Journal of Open Source Education (2018) — https://github.com/barbagroup/CFDPython

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Navier–Stokes incompressible 2D sur le tore [0,1]², méthode de projection (Chorin)
# Grille MAC (décalée) : p au centre des cellules, u sur les faces gauches, v sur les faces du bas.
# Tableaux indexés [j, i] : j ↔ y (lignes), i ↔ x (colonnes).

N  = 128                              # points par direction
nu = 1e-4                             # viscosité
T  = 2.0                              # temps final
h  = 1 / N                            # pas d'espace
dt = min(0.5 * h, 0.2 * h**2 / nu)    # CFL ~ 0.5 (vitesses ~ 1) + diffusion explicite stable
n_steps = int(np.ceil(T / dt))
every = 4                             # une image de l'animation tous les `every` pas de temps

# --- voisins sur le tore : np.roll recolle les bords, c'est toute la périodicité
def xp(f): return np.roll(f, -1, axis=1)   # f(x + h)
def xm(f): return np.roll(f,  1, axis=1)   # f(x - h)
def yp(f): return np.roll(f, -1, axis=0)   # f(y + h)
def ym(f): return np.roll(f,  1, axis=0)   # f(y - h)

def lap(f):                                # laplacien 5 points
    return (xp(f) + xm(f) + yp(f) + ym(f) - 4 * f) / h**2

def div(u, v):                             # divergence, au centre des cellules
    return (xp(u) - u + yp(v) - v) / h

def vort(u, v):                            # vorticité ∂v/∂x - ∂u/∂y, aux coins
    return (v - xm(v)) / h - (u - ym(u)) / h

def interp(f, x, y):                       # interpolation bilinéaire périodique, (x, y) en indices
    i0, j0 = np.floor(x).astype(int), np.floor(y).astype(int)
    a, b = x - i0, y - j0
    i0, j0 = i0 % N, j0 % N
    i1, j1 = (i0 + 1) % N, (j0 + 1) % N
    return (1-a)*(1-b)*f[j0, i0] + a*(1-b)*f[j0, i1] + (1-a)*b*f[j1, i0] + a*b*f[j1, i1]

# --- Poisson Δp = r : sur le tore, le laplacien 5 points est diagonal en Fourier
kx = 2 * np.pi * np.fft.rfftfreq(N)    # axe x : demi-spectre (0 … N/2), comme rfft2
ky = 2 * np.pi * np.fft.fftfreq(N)     # axe y : spectre complet
lam = (2*np.cos(kx)[None, :] + 2*np.cos(ky)[:, None] - 4) / h**2   # forme (N, N//2 + 1)   # ses valeurs propres
lam[0, 0] = 1                          # mode constant : p défini à une constante près
def poisson(r):
    p_hat = np.fft.rfft2(r) / lam
    p_hat[0, 0] = 0
    return np.fft.irfft2(p_hat,s=r.shape)

# --- condition initiale : double couche de cisaillement + petite perturbation (Kelvin–Helmholtz)
I, J = np.meshgrid(np.arange(N), np.arange(N))        # indices de grille
y_u = (J + 0.5) * h                                   # ordonnée des points u
u = np.where(y_u < 0.5, np.tanh(30 * (y_u - 0.25)), np.tanh(30 * (0.75 - y_u)))
v = 0.05 * np.sin(2 * np.pi * (I + 0.5) * h)          # abscisse des points v

frames = [(0.0, vort(u, v))]
for n in range(1, n_steps + 1):
    # 1) advection par les caractéristiques : u(x, t+dt) = u(x - dt·u(x), t)
    v_u = 0.25 * (v + yp(v) + xm(v) + xm(yp(v)))      # v moyenné aux points u
    u_v = 0.25 * (u + xp(u) + ym(u) + ym(xp(u)))      # u moyenné aux points v
    u, v = (interp(u, I - dt*u/h,   J - dt*v_u/h),
            interp(v, I - dt*u_v/h, J - dt*v/h))
    # 2) diffusion (explicite)
    u, v = u + nu*dt*lap(u), v + nu*dt*lap(v)
    # 3) projection : Δp = ∇·u*/dt, puis u = u* - dt ∇p  →  ∇·u = 0
    p = poisson(div(u, v) / dt)
    u, v = u - dt*(p - xm(p))/h, v - dt*(p - ym(p))/h
    if n % every == 0:
        frames.append((n * dt, vort(u, v)))

print(f"{n_steps} pas de temps, {len(frames)} images, max |div u| = {np.abs(div(u, v)).max():.1e}")

# --- animation
w = np.abs(frames[0][1]).max()
fig, ax = plt.subplots(figsize=(5, 4.4), dpi=80)
im = ax.imshow(frames[0][1], origin="lower", extent=(0, 1, 0, 1), cmap="RdBu_r", vmin=-w, vmax=w)
fig.colorbar(im, ax=ax, label="vorticité ω")
ax.set_xticks([]); ax.set_yticks([])

def update(i):
    t, om = frames[i]
    im.set_data(om)
    ax.set_title(f"t = {t:.2f}")
    return [im]

ani = FuncAnimation(fig, update, frames=len(frames), interval=40)   # 40 ms → 25 images/s
plt.close(fig)                        # sinon une image fixe s'affiche en plus de l'animation
# ani.save("kelvin_helmholtz.gif", writer="pillow", fps=25)   # en GIF, pour un README GitHub
HTML(ani.to_jshtml())

512 pas de temps, 129 images, max |div u| = 1.2e-13


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import tqdm
from numba import njit, prange

N  = 200                            
Lx, Ly = 3, 1
Nx, Ny = Lx * N, Ly * N
h  = 1 / N
U0 = 1.0                              
nu = 1e-4                          
T  = 12.0                             
dt = min(0.8 * h / U0, 0.2 * h**2 / nu)
n_steps = int(np.ceil(T / dt))
every = 10                            # une image tous les every pas

D, xc, yc = 0.2, 0.6, 0.5            # carré de côté D, centré en (xc, yc)

# --- voisins sur le tore


def interp(f, x, y):                       # interpolation bilinéaire périodique
    i0, j0 = np.floor(x).astype(int), np.floor(y).astype(int)
    a, b = x - i0, y - j0
    i0, j0 = i0 % Nx, j0 % Ny
    i1, j1 = (i0 + 1) % Nx, (j0 + 1) % Ny
    return (1-a)*(1-b)*f[j0, i0] + a*(1-b)*f[j0, i1] + (1-a)*b*f[j1, i0] + a*b*f[j1, i1]


kx, ky = 2 * np.pi * np.fft.rfftfreq(Nx), 2 * np.pi * np.fft.fftfreq(Ny)
lam = (2*np.cos(kx)[None, :] + 2*np.cos(ky)[:, None] - 4) / h**2
lam[0, 0] = 1
def poisson(r):
    p_hat = np.fft.rfft2(r) / lam
    p_hat[0, 0] = 0
    return np.fft.irfft2(p_hat, s=r.shape)

# --- positions des inconnues, obstacle et éponge
I, J = np.meshgrid(np.arange(Nx), np.arange(Ny))
x_u, y_u = I * h, (J + 0.5) * h
x_v, y_v = (I + 0.5) * h, J * h
def in_square(x, y): return (np.abs(x - xc) <= D / 2) & (np.abs(y - yc) <= D / 2)
solid_u, solid_v = in_square(x_u, y_u), in_square(x_v, y_v)


u = np.full((Ny, Nx), U0)
v = np.zeros((Ny, Nx)) #0.3 * U0 * np.exp(-((x_v - xc - D)**2 + (y_v - yc)**2) / D**2)    pichenette latérale derrière le carré
u[solid_u] = 0


def make_frames(solid,u=u, v=v):
    solid_u, solid_v = solid
    frames_vort = []

    for n in tqdm.tqdm(range(1, n_steps + 1)):
        # 1) advection par les caractéristiques
        v_u = 0.25 * (v + yp(v) + xm(v) + xm(yp(v)))
        u_v = 0.25 * (u + xp(u) + ym(u) + ym(xp(u)))
        u, v = (interp(u, I - dt*u/h,   J - dt*v_u/h),
                interp(v, I - dt*u_v/h, J - dt*v/h))
        # 2) diffusion
        u, v = u + nu*dt*lap(u), v + nu*dt*lap(v)
        # 3) Définition des bords (retour vers U0) et obstacle (vitesse nulle dans le carré)
        u[:,0:10],v[:,0:10]=U0*np.ones_like(u[:,0:10]), np.zeros_like(v[:,0:10])  # condition amont (vent constant): u=U0 à gauche
        Fx = u[solid_u].sum() * h**2 / dt
        u[solid_u], v[solid_v] = 0, 0
        # 4) projection, répétée : remettre le carré à 0 recrée un peu de divergence au bord,
        #    chaque passe en enlève une bonne partie
        for _ in range(3):
            p = poisson(div(u, v) / dt)
            u, v = u - dt*(p - xm(p))/h, v - dt*(p - ym(p))/h
            Fx += u[solid_u].sum() * h**2 / dt
            u[solid_u], v[solid_v] = 0, 0
        if n % every == 0:
            frames_vort.append((n * dt, vort(u, v)))
    return frames_vort
frames = make_frames((solid_u, solid_v))
print(f"{n_steps} pas de temps, {len(frames)} images, Re = U0·D/ν = {U0 * D / nu:.0f}")

# --- animation
w = 15
fig, ax = plt.subplots(figsize=(8, 4.4), dpi=80)
im = ax.imshow(frames[0][1], origin="lower", extent=(0, Lx, 0, Ly), cmap="RdBu_r", vmin=-w, vmax=w)
fig.colorbar(im, ax=ax, label="vorticité ω", shrink=0.8)
ax.set_xticks([]); ax.set_yticks([])

def update(i):
    t, om = frames[i]
    im.set_data(om)
    ax.set_title(f"t = {t:.2f}")
    return [im]

ani = FuncAnimation(fig, update, frames=len(frames), interval=40)
plt.close(fig)
HTML(ani.to_jshtml())

  3%|▎         | 79/3000 [00:08<04:58,  9.77it/s]


KeyboardInterrupt: 

In [ ]:
u=np.array( [[1,2,3,4,5],[1,2,3,4,5]])
u[]

array([1, 1])

In [ ]:
!pip install tqdm

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Alexis\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip
